In [ ]:
import os
import cv2
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
from skimage.feature import local_binary_pattern
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import learning_curve

In [ ]:
def preprocess(img):
    img=cv2.resize(img,(5,5))
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    hsv=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
    gray=cv2.equalizeHist(gray)
    img=cv2.medianBlur(img,5)
    return img,hsv,gray

In [ ]:
def extract_features(img, hsv, gray):

    features = []

    # ========= 1. COLOR =========

    hist = cv2.calcHist([hsv], [0], None, [32], [0,180])
    hist = hist / (hist.sum()+1e-8)
    features.extend(hist.flatten())

    # ========= 2. TEXTURE =========

    lbp = local_binary_pattern(gray, 8, 1, method='uniform')
    lbp_hist, _ = np.histogram(lbp, bins=10, range=(0,10))
    lbp_hist = lbp_hist / (lbp_hist.sum()+1e-8)
    features.extend(lbp_hist)

    # ========= 3. SHAPE =========

    edges = cv2.Canny(gray,100,200)
    contours, _ = cv2.findContours(
        edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )
    area = 0
    perimeter = 0
    for c in contours:
        area += cv2.contourArea(c)
        perimeter += cv2.arcLength(c, True)
    circularity = (4 * np.pi * area) / (perimeter**2 + 1e-6)
    features.extend([area, perimeter, circularity])

    return np.array(features)

In [ ]:
X=[]
y=[]

num=0
base_url="/kaggle/input/datasets/kipshidze/apple-vs-orange-binary-classification/fruit-dataset/apple"
for i in os.listdir(base_url):
    url = os.path.join(base_url, i)
    try:
        img=cv2.imread(url)
        img,hsv,gray=preprocess(img)
        features=extract_features(img,hsv,gray)
        X.append(features)
        y.append(1)
        per=int((len(os.listdir(base_url))-num)/len(os.listdir(base_url))*10)
        print("["+'='*(100-per),' '*per+']',(100-per),'%',end="\r")
        num=num+1
    except Exception as e:
        print("Corrupt Image",e)
print("\nLabel 1 Done")


num=0
base_url="/kaggle/input/datasets/kipshidze/apple-vs-orange-binary-classification/fruit-dataset/orange"
for i in os.listdir(base_url):
    url = os.path.join(base_url, i)
    try:
        img=cv2.imread(url)
        img,hsv,gray=preprocess(img)
        features=extract_features(img,hsv,gray)
        X.append(features)
        y.append(2)
        per=int((len(os.listdir(base_url))-num)/len(os.listdir(base_url))*10)
        print("["+'='*(100-per),' '*per+']',(100-per),'%',end="\r")
        num=num+1
    except:
        print("\nCorrupt Image")
print("\nLabel 2 Done")

In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42,stratify=y_resampled
)
k_values=[3,4,5,6,7,8,10,12]

In [ ]:
plt.figure(figsize=(16, 8))

for k in k_values:
    knn_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(
            n_neighbors=k,
            weights="distance",
        ))
    ])

    train_sizes, train_scores, val_scores = learning_curve(
        estimator=knn_pipeline,
        X=X_resampled,
        y=y_resampled,
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    train_mean = np.mean(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)

    plt.plot(
        train_sizes,
        val_mean,
        label=f"k = {k}"
    )

plt.xlabel("Training Set Size")
plt.ylabel("Validation Accuracy")
plt.title("Learning Curves of k-NN for Different k Values")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
k = 4

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=k, weights="distance",
            metric="minkowski"))
])
knn_pipeline.fit(X_train, y_train)
y_pred = knn_pipeline.predict(X_test)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Apples", "Tomatoes"]))

In [1]:
import os
import cv2
import skimage 
import sklearn 
import imblearn
import numpy as np
import pandas as pd

In [2]:
def preprocess(img):
    img=cv2.resize(img,(128,128))
    
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    gray=cv2.equalizeHist(gray)
    
    hsv=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
    
    img=cv2.medianBlur(img,5)
    return img,hsv,gray

In [3]:
def extract_features(img,hsv,gray):
    feature=[]
    
    # Texture
    lbp=skimage.feature.local_binary_pattern(gray,8,1)
    lbp_hist,_=np.histogram(lbp)
    lbp_hist=lbp_hist.astype(np.float32)
    lbp_hist/=lbp_hist.sum()
    feature.extend(lbp_hist.flatten())

    # Color
    hist=cv2.calcHist([hsv],[0],None,[32],[0,180])
    hist=hist.astype(np.float32)
    hist/=(hist.sum()+1e-8)
    feature.extend(hist.flatten())

    # Shape
    edges=cv2.Canny(gray,150,250)
    contours,_=cv2.findContours(
        edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )
    area=0
    perimeter=0
    for c in contours:
        area+=cv2.contourArea(c)
        perimeter+=cv2.arcLength(c,True)
    circularity=(area*np.pi)/(perimeter**2+1e-9)

    feature.extend([area,perimeter,circularity])

    return feature       

In [4]:
X=[]
y=[]

base_path='/kaggle/input/datasets/kipshidze/apple-vs-orange-binary-classification/fruit-dataset/apple'

for i in os.listdir(base_path):
    try:
        path=os.path.join(base_path,i)
        img=cv2.imread(path)
        img,hsv,gray=preprocess(img)
        features=extract_features(img,hsv,gray)
        X.append(features)
        y.append(0)
    except Exception as e:
        print(e,end='\r')

print()
base_path='/kaggle/input/datasets/kipshidze/apple-vs-orange-binary-classification/fruit-dataset/orange'

for i in os.listdir(base_path):
    try:
        path=os.path.join(base_path,i)
        img=cv2.imread(path)
        img,hsv,gray=preprocess(img)
        features=extract_features(img,hsv,gray)
        X.append(features)
        y.append(1)
    except Exception as e:
        print(e,end='\r')

In [5]:
X=np.array(X,dtype='float32')
y=np.array(y)

In [11]:
k_values=[3,4,5,6,7,8,9,10,11]
best_pipe=None
max_score=0

for k in k_values:
    pipe=sklearn.pipeline.Pipeline([
        ('scale',sklearn.preprocessing.StandardScaler()),
        ('knn',sklearn.neighbors.KNeighborsClassifier(n_neighbors=k))
    ])
    pipe.fit(X,y)
    score=sklearn.metrics.accuracy_score(pipe.predict(X),y)
    if score>max_score:
        max_score=score
        best_pipe=pipe
print(int(max_score*100),"%")

85 %
